In [4]:
import pandas as pd

pd.read_hdf('./data_base_GES1point5/data_base_GES1point5.hdf5')

,category,subcategory,subsubcategory,unit,name,year,co2,ch4,n2o,other,total,uncertainty,ef.unit
0,Électricité,FR,NaN,kWh,Electricité France continentale,2022,0.0,0.0,0.0,0,0.0520,0.10,kg eCO2/kWh
1,Électricité,FR.94,NaN,kWh,Electricité Corse,2014,0.0,0.0,0.0,0,0.5937,0.15,kg eCO2/kWh
2,Électricité,PM,NaN,kWh,Electricité St Pierre et Miquelon,2017,0.0,0.0,0.0,0,0.9438,0.15,kg eCO2/kWh
3,Électricité,WF,NaN,kWh,Electricité Wallis-et-Futuna (identique à St P...,2017,0.0,0.0,0.0,0,0.9438,0.15,kg eCO2/kWh
4,Électricité,TF,NaN,kWh,Electricité Terres australes françaises (ident...,2017,0.0,0.0,0.0,0,0.9438,0.15,kg eCO2/kWh
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1635,Véhicules,bus,bus.intercity,km,Autocar - trajets intercités,2021,0.0,0.0,0.0,0,0.0306,0.60,kg eCO2/km
1636,Véhicules,bus,Bus > 250 000 habitants,km,Autobus moyen - Agglomération de plus de 250 0...,2021,0.0,0.0,0.0,0,0.1290,0.60,kg eCO2/km
1637,Véhicules,bus,Bus 100 000 - 250 000 habitants,km,Autobus moyen - Agglomération de 100 000 à 250...,2021,0.0,0.0,0.0,0,0.1370,0.60,kg eCO2/km
1638,Véhicules,bus,Bus < 100 000 habitants,km,Autobus moyen - Agglomération moins de 100 000...,2021,0.0,0.0,0.0,0,0.1460,0.60,kg eCO2/km


In [ ]:
"""
Convertit le fichier Excel PER1p5 (NACRES–EF) en un HDF5 exploitable par LABeCO2,
en conservant *toutes* les colonnes utiles (même si non utilisées tout de suite).

- Lit l’onglet "NACRES-EF" (et garde l’onglet README à part si besoin)
- Nettoie les noms de colonnes, force les types "code" en str
- Produit 2 tables HDF5 :
    1) /raw_nacres_ef  : table quasi brute (nettoyée)
    2) /purchases_factors : table "compat" proche de ton TSV historique (Achats)
- Exporte aussi 2 TSV (optionnel mais pratique pour diff/inspection)

Usage:
    python convert_per1p5_excel_to_hdf5.py
"""

from __future__ import annotations

from pathlib import Path
import pandas as pd


# --------------------------
# PARAMÈTRES (à adapter)
# --------------------------
EXCEL_PATH = Path("/Users/souchaud/Desktop/dataverse_files/PER1p5_nacres_fe_database_v1-0-2023.xlsx")
OUT_DIR = Path(".")  # mets ici ton dossier data_base_GES1point5/data_initiales/ par ex.
OUT_H5 = OUT_DIR / "GES1point5_purchases_factors_PER1p5_v1-0-2023.h5"
OUT_TSV_RAW = OUT_DIR / "PER1p5_nacres_ef_raw_v1-0-2023.tsv"
OUT_TSV_COMPAT = OUT_DIR / "GES1point5_purchases_factors_PER1p5_compat_v1-0-2023.tsv"

SHEET_DATA = "NACRES-EF"   # onglet qui contient la table
SHEET_README = "README"    # facultatif

DEFAULT_YEAR = 2019
DEFAULT_UNIT = "euro"
DEFAULT_EF_UNIT = "kg eCO2/euro"


# --------------------------
# OUTILS
# --------------------------
def _clean_colnames(cols) -> list[str]:
    # On garde la sémantique d’origine, mais on supprime espaces parasites.
    # (On évite de remplacer '.' par '_' car ton code peut s’appuyer sur les noms exacts.)
    return [str(c).strip() for c in cols]


def _force_str(df: pd.DataFrame, cols: list[str]) -> None:
    for c in cols:
        if c in df.columns:
            df[c] = df[c].astype("string").str.strip()


def _to_float(df: pd.DataFrame, cols: list[str]) -> None:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")


def _category_fr_from_per1p5(cat: str) -> str:
    # Mapping "category" (pour data analysis) -> libellé FR proche de ton TSV
    mapping = {
        "lab.life": "Vie du laboratoire (Alimentation, aménagement, loisirs, bâtiment)",
        "services": "Services",
        "transport": "Transport / Hébergement",
        "consumables": "Consommables (Matières premières, produits chimiques/biologiques et organismes vivants)",
        "lab.equipment": "Matériel et instruments de laboratoire",
        "maintenance": "Réparations et maintenance",
        "info": "Informatique-audiovisuel",
    }
    if cat is None or (isinstance(cat, float) and pd.isna(cat)):
        return "Autre"
    cat = str(cat).strip()
    return mapping.get(cat, cat)


# --------------------------
# PIPELINE
# --------------------------
def main() -> None:
    OUT_DIR.mkdir(parents=True, exist_ok=True)

    # --- Lecture Excel
    xls = pd.ExcelFile(EXCEL_PATH)
    if SHEET_DATA not in xls.sheet_names:
        raise ValueError(f"Onglet '{SHEET_DATA}' introuvable. Onglets: {xls.sheet_names}")

    df = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_DATA, dtype=object)
    df.columns = _clean_colnames(df.columns)

    # --- Nettoyage minimal
    # Colonnes clés (codes & descriptions)
    str_cols = [
        "nacres.code",
        "nacres.description.fr",
        "nacres.description.en",
        "method",
        "module",
        "category",
        "meso.code",
        "meso.description",
        "micro.code",
        "micro.description",
        "ceda.code",
        "ceda.description",
        "ademe.code",
        "ademe.description",
        "useeio.code",
        "useeio.description",
        "uncertainty.groupby",
        "scope.source.v4",
        "scope.source.v5",
    ]
    _force_str(df, str_cols)

    # Colonnes numériques importantes
    num_cols = [
        "per1p5.ef.kg.co2e.per.euro",
        "per1p5.uncertainty.attr.kg.co2e.per.euro",
        "per1p5.uncertainty.80pct.kg.co2e.per.euro",
        "per1p5macro.ef.kg.co2e.per.euro",
        "per1p5macro.uncertainty.attr.kg.co2e.per.euro",
        "per1p5macro.uncertainty.80pct.kg.co2e.per.euro",
        "meso.ef.kg.co2e.per.euro",
        "meso.uncertainty.kg.co2e.per.euro",
        "micro.ef.kg.co2e.per.euro",
        "micro.uncertainty.kg.co2e.per.euro",
        "ademe.ef.kg.co2e.per.euro",
        "ademe.uncertainty.attr.kg.co2e.per.euro",
        "ademe.uncertainty.80pct.kg.co2e.per.euro",
        "useeio.ef.kg.co2e.per.euro",
        "useeio.uncertainty.attr.kg.co2e.per.euro",
        "useeio.uncertainty.80pct.kg.co2e.per.euro",
        "ceda.ef.kg.co2e.per.euro",
        "ceda.uncertainty.attr.kg.co2e.per.euro",
        "ceda.uncertainty.80pct.kg.co2e.per.euro",
    ]
    _to_float(df, num_cols)

    # On supprime les lignes sans code NACRES (si jamais)
    df = df[df["nacres.code"].notna() & (df["nacres.code"].str.len() > 0)].copy()

    # --- Export TSV brut (utile pour inspection)
    df.to_csv(OUT_TSV_RAW, sep="\t", index=False)

    # --- Construction d’une table "compat" proche de ton TSV historique
    # Objectif: pouvoir brancher ton code existant (qui attend: category/subcategory/subsubcategory/unit/name/year/.../total/uncertainty/ef.unit)
    compat = pd.DataFrame()
    compat["category"] = "Achats"

    # Subcategory : on se base sur la colonne "category" (lab.life, consumables, etc.) du dataset PER1p5
    compat["subcategory"] = df["category"].apply(_category_fr_from_per1p5)

    # subsubcategory : ici, le code NACRES (clé stable)
    compat["subsubcategory"] = df["nacres.code"]

    compat["unit"] = DEFAULT_UNIT

    # name : description FR par défaut (tu peux aussi garder EN dans une autre colonne)
    compat["name"] = df["nacres.description.fr"].fillna(df["nacres.description.en"])

    compat["year"] = DEFAULT_YEAR

    # Ton ancien TSV détaillait co2/ch4/n2o/other à 0 : on garde cette structure.
    compat["co2"] = 0.0
    compat["ch4"] = 0.0
    compat["n2o"] = 0.0
    compat["other"] = 0.0

    # total : facteur final arbitrée PER1p5
    compat["total"] = df["per1p5.ef.kg.co2e.per.euro"]

    # uncertainty : on prend par défaut l’incertitude "80%" (valeur absolue en kgCO2e/€)
    # (Tu gardes aussi l’attr. et toutes les autres dans le /raw)
    if "per1p5.uncertainty.80pct.kg.co2e.per.euro" in df.columns:
        compat["uncertainty"] = df["per1p5.uncertainty.80pct.kg.co2e.per.euro"]
    else:
        compat["uncertainty"] = pd.NA

    compat["ef.unit"] = DEFAULT_EF_UNIT

    # Colonnes extra (super utiles pour affichage/audit, sans casser l’ancien format)
    compat["method"] = df["method"]
    compat["module"] = df["module"]
    compat["nacres.description.en"] = df["nacres.description.en"]

    # On garde aussi les candidats macro/meso/micro pour debug & futur UI
    for c in [
        "per1p5macro.ef.kg.co2e.per.euro",
        "meso.ef.kg.co2e.per.euro",
        "micro.ef.kg.co2e.per.euro",
        "per1p5.uncertainty.attr.kg.co2e.per.euro",
        "micro.code",
        "micro.description",
        "meso.code",
        "meso.description",
    ]:
        if c in df.columns:
            compat[c] = df[c]

    # --- Export TSV compat
    compat.to_csv(OUT_TSV_COMPAT, sep="\t", index=False)

    # --- Écriture HDF5
    # On écrit 2 tables:
    #   /raw_nacres_ef        : toutes colonnes (nettoyées) => tu ne perds rien
    #   /purchases_factors    : format proche de ton ancien TSV => plug’n’play
    with pd.HDFStore(OUT_H5, mode="w") as store:
        store.put("raw_nacres_ef", df, format="table", data_columns=True)
        store.put("purchases_factors", compat, format="table", data_columns=True)

        # Optionnel: stocker un README en texte si tu veux
        if SHEET_README in xls.sheet_names:
            readme = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_README, dtype=object)
            readme.columns = _clean_colnames(readme.columns)
            store.put("readme", readme, format="table", data_columns=True)

    print("OK ✅")
    print(f"- HDF5 : {OUT_H5}")
    print(f"- TSV brut : {OUT_TSV_RAW}")
    print(f"- TSV compat : {OUT_TSV_COMPAT}")
    print("\nTables HDF5 écrites : /raw_nacres_ef , /purchases_factors (et /readme si présent).")



In [ ]:

if __name__ == "__main__":
    main()